# Optimization on Grassmannians

This notebook demonstrates optimization algorithms on Grassmannian manifolds using the `grasscalc` package.

## Applications

Optimization on Grassmannians appears in:
- **Eigenvalue problems**: Finding dominant eigenspaces
- **Subspace tracking**: Adaptive signal processing
- **Dimensionality reduction**: PCA and variants
- **Computer vision**: Subspace methods for recognition

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from numpy.linalg import norm

np.set_printoptions(precision=4, suppress=True)

from grasscalc.core.random import sample_grassmann
from grasscalc.core.distances import geodesic_distance, chordal_distance
from grasscalc.core.calculus import riemannian_gradient

## 1. Riemannian Gradient

The Riemannian gradient of f: Gr(k,n) → R at a point U is the tangent vector that points in the direction of steepest ascent.

In [ ]:
from grasscalc.core.tangent import is_tangent

rng = np.random.default_rng(42)
n, k = 7, 3

# Target subspace
target = sample_grassmann(k, n, rng)

# Objective: squared chordal distance to target
def objective(U):
    return chordal_distance(U, target) ** 2

# Starting point
U = sample_grassmann(k, n, rng)

# Compute gradient
grad = riemannian_gradient(U, objective)

print(f"Gradient shape: {grad.shape}")
print(f"Gradient is tangent at U: {is_tangent(U, grad)}")
print(f"Gradient norm: {norm(grad, 'fro'):.4f}")

## 2. Gradient Descent

Gradient descent on the Grassmannian follows the negative gradient direction, using a retraction to stay on the manifold.

In [ ]:
from grasscalc.layer1.flows import run_gradient_flow

# Run gradient descent
result = run_gradient_flow(
    U0=U,
    objective=objective,
    dt=0.2,           # Step size
    tol=1e-8,         # Convergence tolerance
    max_steps=100,
    store_trajectory=True
)

print(f"Converged: {result.converged}")
print(f"Iterations: {result.iterations}")
print(f"Initial energy: {result.energies[0]:.6f}")
print(f"Final energy: {result.final_energy:.6f}")
print(f"Final distance to target: {geodesic_distance(result.final_point, target):.6f}")

In [ ]:
# Plot convergence
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.semilogy(result.energies, 'b-', linewidth=2)
ax1.set_xlabel('Iteration')
ax1.set_ylabel('Objective (log scale)')
ax1.set_title('Convergence of Gradient Descent')
ax1.grid(True)

if result.gradient_norms:
    ax2.semilogy(result.gradient_norms, 'r-', linewidth=2)
    ax2.set_xlabel('Iteration')
    ax2.set_ylabel('Gradient Norm (log scale)')
    ax2.set_title('Gradient Norm Decay')
    ax2.grid(True)

plt.tight_layout()
plt.show()

## 3. Conjugate Gradient Method

The conjugate gradient method typically converges faster than steepest descent by using information from previous iterations.

In [ ]:
from grasscalc.layer1.optim import minimize_on_grassmann

# Same problem with CG
U0 = sample_grassmann(k, n, rng)

result_cg = minimize_on_grassmann(
    U0=U0,
    objective=objective,
    method='conjugate_gradient',
    max_iter=100,
    tol=1e-8
)

# Compare with gradient descent
result_gd = minimize_on_grassmann(
    U0=U0,
    objective=objective,
    method='gradient_descent',
    max_iter=100,
    tol=1e-8,
    dt=0.2
)

print("Conjugate Gradient:")
print(f"  Iterations: {result_cg.iterations}, Final energy: {result_cg.final_energy:.2e}")

print("\nGradient Descent:")
print(f"  Iterations: {result_gd.iterations}, Final energy: {result_gd.final_energy:.2e}")

In [ ]:
# Plot comparison
plt.figure(figsize=(8, 5))
plt.semilogy(result_gd.energies, 'b-', label='Gradient Descent', linewidth=2)
plt.semilogy(result_cg.energies, 'r--', label='Conjugate Gradient', linewidth=2)
plt.xlabel('Iteration')
plt.ylabel('Objective (log scale)')
plt.title('GD vs CG Convergence')
plt.legend()
plt.grid(True)
plt.show()

## 4. Trust Region Method

Trust region methods adapt the step size based on how well a local model predicts the objective decrease.

In [ ]:
result_tr = minimize_on_grassmann(
    U0=U0,
    objective=objective,
    method='trust_region',
    max_iter=100,
    tol=1e-8,
    delta=1.0  # Initial trust region radius
)

print("Trust Region:")
print(f"  Iterations: {result_tr.iterations}")
print(f"  Final energy: {result_tr.final_energy:.2e}")
print(f"  Converged: {result_tr.converged}")

## 5. Eigenvalue Optimization (Rayleigh Quotient)

A classic application: finding the dominant eigenspace of a symmetric matrix by minimizing the negative Rayleigh quotient.

$$R(U) = \text{tr}(U^T A U)$$

Minimizing $-R(U)$ finds the subspace spanned by the top k eigenvectors.

In [ ]:
from grasscalc.layer1.objectives import rayleigh_quotient

# Create a symmetric matrix with known eigenstructure
n = 10
k = 3

# Random orthogonal eigenvectors
Q, _ = np.linalg.qr(rng.standard_normal((n, n)))

# Eigenvalues: 10, 9, 8, 7, 6, 5, 4, 3, 2, 1
eigenvalues = np.arange(n, 0, -1).astype(float)
A = Q @ np.diag(eigenvalues) @ Q.T

print(f"Eigenvalues of A: {eigenvalues}")
print(f"\nSum of top {k} eigenvalues: {np.sum(eigenvalues[:k])}")

In [ ]:
# Objective: negative Rayleigh quotient (to maximize)
def neg_rayleigh(U):
    return -rayleigh_quotient(U, A)

# Random starting point
U0 = sample_grassmann(k, n, rng)
print(f"Initial Rayleigh quotient: {rayleigh_quotient(U0, A):.4f}")

# Optimize
result = minimize_on_grassmann(
    U0=U0,
    objective=neg_rayleigh,
    method='gradient_descent',
    max_iter=300,
    dt=0.05,
    tol=1e-10
)

final_rayleigh = rayleigh_quotient(result.final_point, A)
expected = np.sum(eigenvalues[:k])

print(f"\nFinal Rayleigh quotient: {final_rayleigh:.4f}")
print(f"Expected (sum of top {k}): {expected:.4f}")
print(f"Relative error: {abs(final_rayleigh - expected) / expected * 100:.4f}%")

In [ ]:
# Verify: the optimized subspace should align with top eigenvectors
true_eigenspace = Q[:, :k]
found_eigenspace = result.final_point

# Check overlap (should be close to k)
overlap = np.sum((true_eigenspace.T @ found_eigenspace) ** 2)
print(f"Overlap with true eigenspace: {overlap:.6f} (max = {k})")
print(f"Subspace alignment: {overlap / k * 100:.2f}%")

## 6. Gradient Flow Visualization

Let's visualize the optimization trajectory on a 2D embedding.

In [ ]:
# Run with trajectory stored
U0 = sample_grassmann(k, n, rng)

result = minimize_on_grassmann(
    U0=U0,
    objective=neg_rayleigh,
    method='gradient_descent',
    max_iter=200,
    dt=0.1
)

# Compute distances from trajectory points to optimum
optimum = Q[:, :k]  # True top eigenspace
distances = [geodesic_distance(W, optimum) for W in result.trajectory]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(-np.array(result.energies), 'b-', linewidth=2)
ax1.axhline(y=expected, color='r', linestyle='--', label='Optimal')
ax1.set_xlabel('Iteration')
ax1.set_ylabel('Rayleigh Quotient')
ax1.set_title('Rayleigh Quotient Optimization')
ax1.legend()
ax1.grid(True)

ax2.semilogy(distances, 'g-', linewidth=2)
ax2.set_xlabel('Iteration')
ax2.set_ylabel('Distance to Optimum (log)')
ax2.set_title('Convergence to True Eigenspace')
ax2.grid(True)

plt.tight_layout()
plt.show()

## Summary

This notebook demonstrated:

1. **Riemannian gradient**: Computing gradients on the manifold
2. **Gradient descent**: Basic optimization with retractions
3. **Conjugate gradient**: Faster convergence using history
4. **Trust region**: Adaptive step size control
5. **Eigenvalue optimization**: Finding eigenspaces via Rayleigh quotient

The `grasscalc` package provides production-ready implementations of these algorithms for research and applications.